# Checkpoint A3: Intake Validation — Kiểm tra đầu vào hợp đồng

Checkpoint này triển khai hàm `validate_intake()` để kiểm tra xem một hợp đồng có đủ điều kiện để đưa vào pipeline trích xuất hay không.

**Điều kiện đạt:**
- [x] Hàm kiểm tra file không rỗng (>100 ký tự)
- [x] Phát hiện "HỢP ĐỒNG" / "hợp đồng" trong 200 ký tự đầu
- [x] Đếm số bên tham gia ("Bên A", "Bên B")
- [x] Đếm số điều khoản ("ĐIỀU" sections)
- [x] Trả về dict: is_valid, contract_type, parties, section_count, issues[]
- [x] Chạy test trên cả 3 hợp đồng mẫu

**Tiếp theo:** Sau khi pass, chuyển sang Checkpoint B3 (gọi Gemini API).

In [4]:
import re

def validate_intake(contract_text: str) -> dict:
    """Kiểm tra xem hợp đồng có đủ điều kiện để đưa vào pipeline trích xuất.

    Thuật toán kiểm tra 4 tiêu chí:
    1. Độ dài tối thiểu (>100 ký tự) — loại file rỗng hoặc quá ngắn
    2. Chứa từ khóa "HỢP ĐỒNG" trong phần đầu — xác nhận đây là hợp đồng
    3. Có ít nhất 2 bên ("Bên A", "Bên B") — hợp đồng cần 2 bên
    4. Có các "ĐIỀU" sections — cấu trúc hợp đồng tiêu chuẩn VN

    Args:
        contract_text: Nội dung đầy đủ của hợp đồng (chuỗi UTF-8)

    Returns:
        Dict chứa:
          - is_valid (bool): Đủ điều kiện để xử lý tiếp?
          - contract_type (str): Loại hợp đồng phát hiện được
          - parties (list[str]): Danh sách các bên tham gia
          - section_count (int): Số điều khoản đếm được
          - issues (list[str]): Danh sách vấn đề (rỗng nếu hợp lệ)
    """
    issues = []

    # --- Tiêu chí 1: Độ dài tối thiểu ---
    if len(contract_text) <= 100:
        issues.append("File quá ngắn (<=100 ký tự) — có thể file rỗng hoặc lỗi load")

    # --- Tiêu chí 2: Phát hiện từ khóa "HỢP ĐỒNG" trong 200 ký tự đầu ---
    header = contract_text[:200].upper()
    if "HỢP ĐỒNG" not in header:
        issues.append('Không tìm thấy "HỢP ĐỒNG" trong 200 ký tự đầu — có thể không phải hợp đồng')

    # --- Tiêu chí 3: Đếm các bên tham gia ---
    parties = []
    if re.search(r"Bên\s+A", contract_text, re.IGNORECASE):
        parties.append("Bên A")
    if re.search(r"Bên\s+B", contract_text, re.IGNORECASE):
        parties.append("Bên B")
    if re.search(r"Bên\s+C", contract_text, re.IGNORECASE):
        parties.append("Bên C")

    if len(parties) < 2:
        issues.append(f"Chỉ tìm thấy {len(parties)} bên — hợp đồng cần ít nhất 2 bên (Bên A, Bên B)")

    # --- Tiêu chí 4: Đếm điều khoản ---
    sections = re.findall(r"ĐIỀU\s+\d+", contract_text, re.IGNORECASE)
    section_count = len(sections)

    if section_count == 0:
        issues.append('Không tìm thấy "ĐIỀU" nào — cấu trúc không phải hợp đồng tiêu chuẩn VN')

    # --- Xác định loại hợp đồng ---
    contract_type = "unknown"
    text_lower = contract_text.lower()
    type_keywords = {
        "service_agreement": ["cung cấp dịch vụ", "dịch vụ viễn thông", "sla", "vận hành"],
        "lease_agreement": ["thuê", "cho thuê", "mặt bằng"],
        "purchase_contract": ["mua bán", "cung cấp thiết bị", "hàng hóa"],
        "nda": ["bảo mật thông tin", "không tiết lộ", "nda"],
    }
    for ctype, keywords in type_keywords.items():
        if any(kw in text_lower for kw in keywords):
            contract_type = ctype
            break

    # --- Kết quả ---
    is_valid = len(issues) == 0

    return {
        "is_valid": is_valid,
        "contract_type": contract_type,
        "parties": parties,
        "section_count": section_count,
        "issues": issues,
    }


# Quick test với dữ liệu mẫu
sample_valid = """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
Số: HD/2026/001

Bên A: Công ty Viễn thông A
Bên B: Công ty Công nghệ B

ĐIỀU 1: Đối tượng hợp đồng
ĐIỀU 2: Thời hạn hợp đồng
ĐIỀU 3: Giá trị hợp đồng
ĐIỀU 4: Phạt vi phạm SLA
""" * 3  # Nhân lên để >100 ký tự

result = validate_intake(sample_valid)
print("Test mẫu:", result)

Test mẫu: {'is_valid': True, 'contract_type': 'service_agreement', 'parties': ['Bên A', 'Bên B'], 'section_count': 12, 'issues': []}


In [5]:
# ============================================================
# Test validate_intake trên các hợp đồng thực tế đã sao chép ở A1
# ============================================================
import os

try:
    from docx import Document as DocxDocument
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

PROJECT_ROOT = "../../outputs/contract-term-extractor"
contracts_dir = f"{PROJECT_ROOT}/data/contracts"
contracts = {}

if not HAS_DOCX:
    print("HỆ THỐNG THIẾU THƯ VIỆN python-docx. Hãy chạy lệnh dưới đây để cài đặt:")
    print("  !pip install python-docx")
    print("\nTạm thời sử dụng dữ liệu fallback để chạy test in-memory:\n")
    # Fallback dữ liệu text để notebook không bị crash
    contracts = {
        "contract-001.docx": "HỢP ĐỒNG CUNG CẤP DỊCH VỤ TRUYỀN DẪN LIÊN QUẬN\nSố: HD-DV-2026-001\nBên A... Bên B... Điều 1... Điều 2...",
        "contract-002.docx": "HỢP ĐỒNG MUA SẮM THIẾT BỊ MẠNG\nSố: HD-MS-2026-002\n[Lỗi OCR] Bên A...",
        "contract-003-risky.docx": "HỢP ĐỒNG DỊCH VỤ VẬN HÀNH MẠNG\nSố: HD-DV-2026-003\nBên A... Bên B... tự động gia hạn... phạt 5% không giới hạn...",
    }
else:
    if os.path.exists(contracts_dir):
        for fname in os.listdir(contracts_dir):
            if fname.endswith(".docx"):
                fpath = os.path.join(contracts_dir, fname)
                try:
                    doc = DocxDocument(fpath)
                    contracts[fname] = "\n".join(p.text for p in doc.paragraphs)
                except Exception as e:
                    print(f"Lỗi đọc {fname}: {e}")

# Chạy validate trên các hợp đồng và in bảng kết quả
print("=" * 80)
print(f"{'Hợp đồng':<30} {'Hợp lệ':>7} {'Loại':<20} {'Bên':>5} {'ĐIỀU':>5} {'Issues'}")
print("=" * 80)

for name, text in contracts.items():
    result = validate_intake(text)
    valid_str = "YES" if result["is_valid"] else "NO"
    issues_str = "; ".join(result["issues"]) if result["issues"] else "-"
    print(f"{name:<30} {valid_str:>7} {result['contract_type']:<20} {len(result['parties']):>5} {result['section_count']:>5}  {issues_str}")
print("=" * 80)

# Chi tiết cho contract-003-risky.docx (nếu có)
key_detail = "contract-003-risky.docx"
if key_detail in contracts:
    print(f"\nChi tiết {key_detail}:")
    detail = validate_intake(contracts[key_detail])
    for key, value in detail.items():
        print(f"  {key}: {value}")

Hợp đồng                        Hợp lệ Loại                   Bên  ĐIỀU Issues
contract-002.docx                  YES unknown                  2     6  -
contract-003-risky.docx            YES service_agreement        2     9  -
contract-001.docx                  YES service_agreement        3     8  -
contract-004-telecom-sla.docx      YES service_agreement        3    10  -

Chi tiết contract-003-risky.docx:
  is_valid: True
  contract_type: service_agreement
  parties: ['Bên A', 'Bên B']
  section_count: 9
  issues: []


In [6]:
# ============================================================
# Ghi file script python intake.py chuẩn vào thư mục scripts/
# ============================================================
import shutil
import os

PROJECT_ROOT = "../../outputs/contract-term-extractor"
src_script = "../../templates/skills/contract-term-extractor/scripts/intake.py"
dst_script = f"{PROJECT_ROOT}/scripts/intake.py"

os.makedirs(os.path.dirname(dst_script), exist_ok=True)
shutil.copy(src_script, dst_script)
print(f"Đã copy intake.py thành công vào: {dst_script}")

Đã copy intake.py thành công vào: ../../outputs/contract-term-extractor/scripts/intake.py
